# Task 1 - Evaluation and text generation (Aswin)
Run after training, in the same uploaded repository folder. This recomputes held-out metrics, then writes greedy and temperature 0.7/1.0/1.2 samples with prompts, checkpoint identity, timing, and diversity metrics.

In [ ]:
from pathlib import Path
import importlib.util, os, subprocess, sys

packages = {'yaml': 'pyyaml', 'psutil': 'psutil', 'pandas': 'pandas', 'matplotlib': 'matplotlib'}
missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])

def locate_repo_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'task1_llm/member_aswin/configs/full.yaml').exists(): return candidate.resolve()
    content = Path('/content')
    matches = list(content.rglob('task1_llm/member_aswin/configs/full.yaml')) if content.exists() else []
    if not matches:
        try:
            from google.colab import drive
            print('Repository not found in /content; mounting Google Drive ...')
            drive.mount('/content/drive')
            drive_root = Path('/content/drive/MyDrive')
            matches = list(drive_root.rglob('task1_llm/member_aswin/configs/full.yaml'))
        except ImportError:
            pass
    if not matches: raise FileNotFoundError('Upload the complete folder containing task1_llm to /content or My Drive.')
    return matches[0].resolve().parents[3]

REPO_ROOT = locate_repo_root(); os.chdir(REPO_ROOT)
CONFIG = REPO_ROOT / 'task1_llm/member_aswin/configs/full.yaml'
SRC = REPO_ROOT / 'task1_llm/member_aswin/src'
import torch, yaml
RUN_ID = yaml.safe_load(CONFIG.read_text())['run_id']
checkpoint = REPO_ROOT / 'task1_llm/member_aswin/checkpoints' / RUN_ID / 'best_model.pt'
if not checkpoint.exists(): raise FileNotFoundError('Best checkpoint is missing. Run model_training.ipynb first.')
print('Repository root:', REPO_ROOT)
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('Checkpoint:', checkpoint)
subprocess.run([sys.executable, str(SRC / 'evaluate.py'), '--config', str(CONFIG)], cwd=REPO_ROOT, check=True)
subprocess.run([sys.executable, str(SRC / 'generate.py'), '--config', str(CONFIG)], cwd=REPO_ROOT, check=True)

In [ ]:
samples_path = REPO_ROOT / 'task1_llm/member_aswin/outputs/generated_text' / f'{RUN_ID}_samples.txt'
print(samples_path.read_text())

In [ ]:
import pandas as pd
pd.read_csv(REPO_ROOT / 'task1_llm/member_aswin/metrics_report.csv')